<a href="https://colab.research.google.com/github/koushik1601/Koushiksarunreddy_INFO5731_Spring2026/blob/main/Yuvachandran_koushiksarunreddy_Assignment_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [ ]:
# Question 1: Collect 10,000 research paper abstracts from Semantic Scholar
# Using the free Semantic Scholar API (no API key required for basic use)

import requests
import pandas as pd
import time

def fetch_papers(query, total=10000, batch_size=100):
    """Fetch papers from Semantic Scholar API"""
    papers = []
    offset = 0
    fields = "title,abstract,year,authors,venue,externalIds"
    base_url = "https://api.semanticscholar.org/graph/v1/paper/search"

    while len(papers) < total:
        params = {
            "query": query,
            "fields": fields,
            "limit": batch_size,
            "offset": offset
        }

        try:
            response = requests.get(base_url, params=params, timeout=30)

            if response.status_code == 429:
                print("Rate limited. Waiting 60 seconds...")
                time.sleep(60)
                continue

            if response.status_code != 200:
                print(f"Error {response.status_code}: {response.text}")
                break

            data = response.json()
            batch = data.get("data", [])

            if not batch:
                print("No more results found.")
                break

            for paper in batch:
                # Only include papers with abstracts
                if paper.get("abstract"):
                    papers.append({
                        "title": paper.get("title", ""),
                        "abstract": paper.get("abstract", ""),
                        "year": paper.get("year", ""),
                        "venue": paper.get("venue", ""),
                        "authors": ", ".join([a["name"] for a in paper.get("authors", [])]),
                        "paperId": paper.get("paperId", ""),
                        "query": query
                    })

            offset += batch_size
            print(f"Collected {len(papers)} papers so far (offset={offset})...")

            # Be polite to the API
            time.sleep(3)

        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")
            time.sleep(10)

    return papers[:total]

# -----------------------------------------------
# Collect papers using multiple queries
# -----------------------------------------------
all_papers = []
queries = ["machine learning", "data science", "artificial intelligence", "information extraction"]
papers_per_query = 2500  # 4 queries × 2500 = 10,000

for q in queries:
    print(f"\n🔍 Fetching papers for query: '{q}'")
    papers = fetch_papers(q, total=papers_per_query)
    all_papers.extend(papers)
    print(f"✅ Collected {len(papers)} papers for '{q}'")

# -----------------------------------------------
# Save to CSV
# -----------------------------------------------
df = pd.DataFrame(all_papers)

# Remove duplicate papers (same paperId)
df.drop_duplicates(subset="paperId", inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"\n📊 Total unique papers collected: {len(df)}")

# Save raw data
df.to_csv("research_papers_raw.csv", index=False)
print("✅ Saved to 'research_papers_raw.csv'")

# Preview
df.head()



🔍 Fetching papers for query: 'machine learning'
Rate limited. Waiting 60 seconds...
Rate limited. Waiting 60 seconds...
Rate limited. Waiting 60 seconds...
Rate limited. Waiting 60 seconds...
Collected 62 papers so far (offset=100)...
Rate limited. Waiting 60 seconds...
Rate limited. Waiting 60 seconds...
Rate limited. Waiting 60 seconds...
Rate limited. Waiting 60 seconds...
Collected 140 papers so far (offset=200)...
Rate limited. Waiting 60 seconds...
Rate limited. Waiting 60 seconds...
Collected 213 papers so far (offset=300)...
Rate limited. Waiting 60 seconds...
Collected 283 papers so far (offset=400)...
Rate limited. Waiting 60 seconds...
Collected 330 papers so far (offset=500)...
Collected 384 papers so far (offset=600)...
Rate limited. Waiting 60 seconds...
Rate limited. Waiting 60 seconds...
Rate limited. Waiting 60 seconds...
Rate limited. Waiting 60 seconds...
Rate limited. Waiting 60 seconds...
Rate limited. Waiting 60 seconds...
Collected 425 papers so far (offset=700)

,title,abstract,year,venue,authors,paperId,query
0,"Machine Learning: Algorithms, Real-World Appli...",In the current age of the Fourth Industrial Re...,2021,SN Computer Science,Iqbal H. Sarker,7872f34e2a164c5cf3c34a7a7433dc3342b6c7ea,machine learning
1,A Survey on Bias and Fairness in Machine Learning,With the widespread use of artificial intellig...,2019,ACM Computing Surveys,"Ninareh Mehrabi, Fred Morstatter, N. Saxena, K...",0090023afc66cd2741568599057f4e82b566137c,machine learning
2,Towards A Rigorous Science of Interpretable Ma...,"As machine learning systems become ubiquitous,...",2017,,"F. Doshi-Velez, Been Kim",5c39e37022661f81f79e481240ed9b175dec6513,machine learning
3,TensorFlow: A system for large-scale machine l...,TensorFlow is a machine learning system that o...,2016,USENIX Symposium on Operating Systems Design a...,"Martín Abadi, P. Barham, Jianmin Chen, Z. Chen...",4954fa180728932959997a4768411ff9136aac81,machine learning
4,TensorFlow: Large-Scale Machine Learning on He...,TensorFlow is an interface for expressing mach...,2016,arXiv.org,"Martín Abadi, Ashish Agarwal, P. Barham, E. Br...",9c9d7247f8c51ec5a02b0d911d1d7b9e8160495d,machine learning


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [ ]:
# Write code for each of the sub parts with proper comments.
# ============================================================
# Question 2: Clean the text data collected in Question 1
# ============================================================

!pip install nltk pandas

import pandas as pd
import nltk
import re
import string

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Load the raw data
df = pd.read_csv("research_papers_raw.csv")
print(f"Loaded {len(df)} papers")
print(df['abstract'].head(3))

# Initialize tools
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# -----------------------------------------------
# (1) Remove special characters and punctuation
# -----------------------------------------------
def remove_noise(text):
    if pd.isna(text): return ""
    text = re.sub(r'<.*?>', '', text)           # Remove HTML tags
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special chars & punctuation
    text = re.sub(r'\s+', ' ', text).strip()    # Remove extra whitespace
    return text

df['clean_step1'] = df['abstract'].apply(remove_noise)
print("\n✅ Step 1 - Noise removed:")
print(df['clean_step1'].head(2))

# -----------------------------------------------
# (2) Remove numbers
# -----------------------------------------------
def remove_numbers(text):
    return re.sub(r'\d+', '', text).strip()

df['clean_step2'] = df['clean_step1'].apply(remove_numbers)
print("\n✅ Step 2 - Numbers removed:")
print(df['clean_step2'].head(2))

# -----------------------------------------------
# (3) Remove stopwords
# -----------------------------------------------
def remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w.lower() not in stop_words]
    return ' '.join(filtered)

df['clean_step3'] = df['clean_step2'].apply(remove_stopwords)
print("\n✅ Step 3 - Stopwords removed:")
print(df['clean_step3'].head(2))

# -----------------------------------------------
# (4) Lowercase all text
# -----------------------------------------------
df['clean_step4'] = df['clean_step3'].apply(lambda x: x.lower())
print("\n✅ Step 4 - Lowercased:")
print(df['clean_step4'].head(2))

# -----------------------------------------------
# (5) Stemming
# -----------------------------------------------
def apply_stemming(text):
    tokens = word_tokenize(text)
    stemmed = [stemmer.stem(w) for w in tokens]
    return ' '.join(stemmed)

df['clean_step5_stemmed'] = df['clean_step4'].apply(apply_stemming)
print("\n✅ Step 5 - Stemming applied:")
print(df['clean_step5_stemmed'].head(2))

# -----------------------------------------------
# (6) Lemmatization
# -----------------------------------------------
def apply_lemmatization(text):
    tokens = word_tokenize(text)
    lemmatized = [lemmatizer.lemmatize(w) for w in tokens]
    return ' '.join(lemmatized)

df['clean_final_lemmatized'] = df['clean_step4'].apply(apply_lemmatization)
print("\n✅ Step 6 - Lemmatization applied:")
print(df['clean_final_lemmatized'].head(2))

# -----------------------------------------------
# Save cleaned data
# -----------------------------------------------
df.to_csv("research_papers_cleaned.csv", index=False)
print(f"\n✅ Cleaned data saved to 'research_papers_cleaned.csv'")
print(f"Columns: {list(df.columns)}")
df.head(3)



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


Loaded 1669 papers
0    In the current age of the Fourth Industrial Re...
1    With the widespread use of artificial intellig...
2    As machine learning systems become ubiquitous,...
Name: abstract, dtype: object

✅ Step 1 - Noise removed:
0    In the current age of the Fourth Industrial Re...
1    With the widespread use of artificial intellig...
Name: clean_step1, dtype: object

✅ Step 2 - Numbers removed:
0    In the current age of the Fourth Industrial Re...
1    With the widespread use of artificial intellig...
Name: clean_step2, dtype: object

✅ Step 3 - Stopwords removed:
0    current age Fourth Industrial Revolution IR In...
1    widespread use artificial intelligence AI syst...
Name: clean_step3, dtype: object

✅ Step 4 - Lowercased:
0    current age fourth industrial revolution ir in...
1    widespread use artificial intelligence ai syst...
Name: clean_step4, dtype: object

✅ Step 5 - Stemming applied:
0    current age fourth industri revolut ir industr...
1    widespread us

,title,abstract,year,venue,authors,paperId,query,clean_step1,clean_step2,clean_step3,clean_step4,clean_step5_stemmed,clean_final_lemmatized
0,"Machine Learning: Algorithms, Real-World Appli...",In the current age of the Fourth Industrial Re...,2021,SN Computer Science,Iqbal H. Sarker,7872f34e2a164c5cf3c34a7a7433dc3342b6c7ea,machine learning,In the current age of the Fourth Industrial Re...,In the current age of the Fourth Industrial Re...,current age Fourth Industrial Revolution IR In...,current age fourth industrial revolution ir in...,current age fourth industri revolut ir industr...,current age fourth industrial revolution ir in...
1,A Survey on Bias and Fairness in Machine Learning,With the widespread use of artificial intellig...,2019,ACM Computing Surveys,"Ninareh Mehrabi, Fred Morstatter, N. Saxena, K...",0090023afc66cd2741568599057f4e82b566137c,machine learning,With the widespread use of artificial intellig...,With the widespread use of artificial intellig...,widespread use artificial intelligence AI syst...,widespread use artificial intelligence ai syst...,widespread use artifici intellig ai system app...,widespread use artificial intelligence ai syst...
2,Towards A Rigorous Science of Interpretable Ma...,"As machine learning systems become ubiquitous,...",2017,NaN,"F. Doshi-Velez, Been Kim",5c39e37022661f81f79e481240ed9b175dec6513,machine learning,As machine learning systems become ubiquitous ...,As machine learning systems become ubiquitous ...,machine learning systems become ubiquitous sur...,machine learning systems become ubiquitous sur...,machin learn system becom ubiquit surg interes...,machine learning system become ubiquitous surg...


# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [ ]:
# Your code here
# ============================================================
# Question 3: Syntax and Structure Analysis
# ============================================================

!pip install nltk spacy pandas
!python -m spacy download en_core_web_sm

import pandas as pd
import nltk
import spacy
from nltk import pos_tag, word_tokenize, ne_chunk
from nltk.tree import Tree
from collections import Counter

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

nlp = spacy.load("en_core_web_sm")

# Load cleaned data
df = pd.read_csv("research_papers_cleaned.csv")

# Work on a sample of 500 abstracts for performance
sample_df = df[df['abstract'].notna()].head(500).copy()
print(f"Working with {len(sample_df)} abstracts for analysis")

# -----------------------------------------------
# (1) POS Tagging - Count N, V, Adj, Adv
# -----------------------------------------------
print("\n" + "="*60)
print("PART 1: POS TAGGING")
print("="*60)

noun_count = verb_count = adj_count = adv_count = 0

for text in sample_df['clean_final_lemmatized']:
    tokens = word_tokenize(str(text))
    tags = pos_tag(tokens)
    for _, tag in tags:
        if tag.startswith('NN'):  noun_count += 1
        elif tag.startswith('VB'): verb_count += 1
        elif tag.startswith('JJ'): adj_count += 1
        elif tag.startswith('RB'): adv_count += 1

print(f"Total Nouns      (NN) : {noun_count}")
print(f"Total Verbs      (VB) : {verb_count}")
print(f"Total Adjectives (JJ) : {adj_count}")
print(f"Total Adverbs    (RB) : {adv_count}")

# Show POS tags for one example sentence
example = str(sample_df['abstract'].iloc[0])
example_sentence = ' '.join(example.split('.')[:1])
tokens = word_tokenize(example_sentence)
tags = pos_tag(tokens)
print(f"\nExample POS Tags (first sentence of abstract 1):")
print(tags)

# -----------------------------------------------
# (2) Constituency & Dependency Parsing
# -----------------------------------------------
print("\n" + "="*60)
print("PART 2: CONSTITUENCY & DEPENDENCY PARSING")
print("="*60)

# --- Constituency Parsing using NLTK (rule-based grammar) ---
print("\n--- Constituency Parsing (NLTK) ---")

# Define a simple grammar
grammar = nltk.CFG.fromstring("""
  S  -> NP VP
  VP -> V NP | V NP PP | V PP | V
  PP -> P NP
  NP -> DT NN | DT NNS | NN | NNS | DT JJ NN | JJ NN | PRP
  V  -> VB | VBZ | VBP | VBD | VBN | VBG
  DT -> 'the' | 'a' | 'an' | 'this' | 'these' | 'our'
  NN -> 'model' | 'method' | 'network' | 'data' | 'system' | 'paper' | 'result' | 'approach'
  NNS -> 'models' | 'methods' | 'networks' | 'results' | 'systems' | 'papers'
  JJ -> 'new' | 'deep' | 'large' | 'neural' | 'better' | 'high'
  VB -> 'propose' | 'show' | 'present' | 'use' | 'train' | 'learn'
  VBZ -> 'proposes' | 'shows' | 'presents' | 'uses'
  VBP -> 'propose' | 'present' | 'show'
  VBD -> 'proposed' | 'showed' | 'used' | 'trained'
  VBN -> 'proposed' | 'shown' | 'used' | 'trained'
  VBG -> 'proposing' | 'showing' | 'using' | 'training'
  PRP -> 'we' | 'it'
  P  -> 'on' | 'in' | 'for' | 'with' | 'of'
""")

parser = nltk.ChartParser(grammar)
test_sentence = ["we", "propose", "a", "new", "model"]

print(f"\nSentence: {' '.join(test_sentence)}")
trees = list(parser.parse(test_sentence))
if trees:
    for tree in trees[:1]:
        print("Constituency Parse Tree:")
        print(tree)
        tree.pretty_print()
else:
    print("No parse found with simplified grammar.")

# --- Dependency Parsing using spaCy ---
print("\n--- Dependency Parsing (spaCy) ---")

example_text = "We propose a novel deep learning model for text classification."
doc = nlp(example_text)

print(f"\nSentence: {example_text}")
print(f"\n{'Token':<20} {'Dep Label':<20} {'Head':<20} {'POS'}")
print("-" * 70)
for token in doc:
    print(f"{token.text:<20} {token.dep_:<20} {token.head.text:<20} {token.pos_}")

# Print dependency trees for all sentences in first 5 abstracts
print("\n\n--- Dependency Trees for First 5 Abstracts ---")
for i, abstract in enumerate(sample_df['abstract'].head(5)):
    print(f"\n[Abstract {i+1}]")
    doc = nlp(str(abstract)[:500])  # Limit to 500 chars for readability
    for sent in doc.sents:
        print(f"\n  Sentence: {sent.text[:100]}...")
        for token in sent:
            print(f"    {token.text:<15} --[{token.dep_}]--> {token.head.text}")

# --- Explanation ---
print("\n" + "="*60)
print("EXPLANATION: Constituency vs Dependency Parsing")
print("="*60)
print("""
CONSTITUENCY PARSING TREE:
  Breaks a sentence into nested sub-phrases (constituents).
  Example: "We propose a new model"
    S
    ├── NP (We)
    └── VP
        ├── V (propose)
        └── NP
            ├── DT (a)
            ├── JJ (new)
            └── NN (model)
  → Shows the hierarchical phrase structure of the sentence.
    Each node is a grammatical unit (S=Sentence, NP=Noun Phrase, etc.)

DEPENDENCY PARSING TREE:
  Shows binary grammatical relationships between words (head → dependent).
  Example: "We propose a new model"
    propose (ROOT)
    ├── We      (nsubj - nominal subject)
    ├── model   (dobj  - direct object)
        ├── a   (det   - determiner)
        └── new (amod  - adjectival modifier)
  → Shows HOW words relate to each other rather than phrase groupings.
    More useful for information extraction and NLP tasks.
""")

# -----------------------------------------------
# (3) Named Entity Recognition (NER)
# -----------------------------------------------
print("="*60)
print("PART 3: NAMED ENTITY RECOGNITION (NER)")
print("="*60)

entity_counts = Counter()
entity_examples = {}

for text in sample_df['abstract'].head(300):
    doc = nlp(str(text)[:1000])
    for ent in doc.ents:
        label = ent.label_
        entity_counts[label] += 1
        if label not in entity_examples:
            entity_examples[label] = ent.text

print(f"\nEntity Type Counts across 300 abstracts:")
print(f"\n{'Entity Type':<15} {'Count':<10} {'Example'}")
print("-" * 55)
for ent_type, count in entity_counts.most_common():
    example = entity_examples.get(ent_type, "")
    print(f"{ent_type:<15} {count:<10} {example}")

# Show entities from one example abstract
print("\n\nExample NER on Abstract 1:")
doc = nlp(str(sample_df['abstract'].iloc[0]))
print(f"Text: {str(sample_df['abstract'].iloc[0])[:300]}...\n")
print(f"{'Entity Text':<30} {'Label':<15} {'Description'}")
print("-" * 65)
for ent in doc.ents:
    print(f"{ent.text:<30} {ent.label_:<15} {spacy.explain(ent.label_)}")



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 75.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


Working with 500 abstracts for analysis

PART 1: POS TAGGING
Total Nouns      (NN) : 27366
Total Verbs      (VB) : 9784
Total Adjectives (JJ) : 11780
Total Adverbs    (RB) : 2505

Example POS Tags (first sentence of abstract 1):
[('In', 'IN'), ('the', 'DT'), ('current', 'JJ'), ('age', 'NN'), ('of', 'IN'), ('the', 'DT'), ('Fourth', 'NNP'), ('Industrial', 'NNP'), ('Revolution', 'NNP'), ('(', '('), ('4IR', 'CD'), ('or', 'CC'), ('Industry', 'NNP'), ('4', 'CD')]

PART 2: CONSTITUENCY & DEPENDENCY PARSING

--- Constituency Parsing (NLTK) ---

Sentence: we propose a new model
Constituency Parse Tree:
(S
  (NP (PRP we))
  (VP (V (VBP propose)) (NP (DT a) (JJ new) (NN model))))
             S               
  ___________|___             
 |               VP          
 |      _________|___         
 NP    V             NP      
 |     |      _______|____    
PRP   VBP    DT      JJ   NN 
 |     |     |       |    |   
 we propose  a      new model


--- Dependency Parsing (spaCy) ---

Sentence: 

# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

In [ ]:
# ============================================================
# Question 4: GitHub Marketplace Actions Scraper
# ============================================================

!pip install requests beautifulsoup4 pandas nltk lxml

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.stem import WordNetLemmatizer

# -----------------------------------------------
# PART 1: Scrape GitHub Marketplace Actions
# -----------------------------------------------

BASE_URL = "https://github.com/marketplace"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

def scrape_github_marketplace(max_products=1000):
    all_products = []
    page = 1

    while len(all_products) < max_products:
        url = f"{BASE_URL}?type=actions&page={page}"
        print(f"Scraping page {page} | Products so far: {len(all_products)}")

        try:
            response = requests.get(url, headers=headers, timeout=30)
            if response.status_code != 200:
                print(f"Status {response.status_code} on page {page}. Stopping.")
                break
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(10)
            continue

        soup = BeautifulSoup(response.text, "lxml")

        # Find all marketplace action cards
        # GitHub uses article tags for each listing
        cards = soup.find_all("article", class_=re.compile(r"marketplace", re.I))
        if not cards:
            # Fallback: look for any article or div with listing info
            cards = soup.find_all("div", class_=re.compile(r"col-md-4|d-flex.*border", re.I))

        if not cards:
            print(f"No cards found on page {page}. Trying alternative selector...")
            cards = soup.select("div.d-flex.flex-column.border")

        if not cards:
            print(f"No more products found. Stopping at page {page}.")
            break

        for card in cards:
            # Product name
            name = ""
            name_tag = card.find(["h3", "h4", "a"], class_=re.compile(r"h4|f3|name|title", re.I))
            if not name_tag:
                name_tag = card.find("a")
            if name_tag:
                name = name_tag.get_text(strip=True)

            # Description
            description = ""
            desc_tag = card.find("p")
            if desc_tag:
                description = desc_tag.get_text(strip=True)

            # URL
            url_tag = card.find("a", href=True)
            product_url = ""
            if url_tag:
                href = url_tag.get("href", "")
                product_url = f"https://github.com{href}" if href.startswith("/") else href

            if name:
                all_products.append({
                    "product_name": name,
                    "description": description,
                    "url": product_url,
                    "page_number": page
                })

        print(f"  ✓ Found {len(cards)} products on page {page}")

        # Checkpoint save every 100 products
        if len(all_products) % 100 == 0 and len(all_products) > 0:
            pd.DataFrame(all_products).to_csv("github_marketplace_checkpoint.csv", index=False)
            print(f"  💾 Checkpoint saved: {len(all_products)} products")

        page += 1
        time.sleep(2)  # Polite delay

    return all_products[:max_products]

print("="*60)
print("PART 1: Scraping GitHub Marketplace Actions...")
print("="*60)
products = scrape_github_marketplace(max_products=1000)

df_github = pd.DataFrame(products)
df_github.to_csv("github_marketplace_raw.csv", index=False)
print(f"\n✅ Saved {len(df_github)} products to 'github_marketplace_raw.csv'")
print(df_github.head(5))

# -----------------------------------------------
# PART 2: Preprocessing & Data Quality
# -----------------------------------------------

print("\n" + "="*60)
print("PART 2: PREPROCESSING & DATA QUALITY")
print("="*60)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# --- Preprocessing ---
def preprocess_text(text):
    if pd.isna(text) or text == "": return ""
    text = re.sub(r'<.*?>', '', text)              # Remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)        # Remove special chars & numbers
    text = text.lower().strip()                     # Lowercase
    tokens = word_tokenize(text)                    # Tokenize
    tokens = [t for t in tokens if t not in stop_words]   # Remove stopwords
    tokens = [lemmatizer.lemmatize(t) for t in tokens]    # Lemmatize
    return ' '.join(tokens)

df_github['cleaned_name'] = df_github['product_name'].apply(preprocess_text)
df_github['cleaned_description'] = df_github['description'].apply(preprocess_text)

print("✅ Preprocessing complete")
print(df_github[['product_name', 'cleaned_name', 'description', 'cleaned_description']].head(5))

# --- Data Quality Checks ---
print("\n--- Data Quality Report ---")
print(f"Total records          : {len(df_github)}")
print(f"Duplicate rows         : {df_github.duplicated().sum()}")
print(f"Duplicate URLs         : {df_github['url'].duplicated().sum()}")
print(f"Missing product names  : {df_github['product_name'].isna().sum()}")
print(f"Missing descriptions   : {df_github['description'].isna().sum() + (df_github['description'] == '').sum()}")
print(f"Missing URLs           : {df_github['url'].isna().sum() + (df_github['url'] == '').sum()}")

# Fix: Remove duplicates
df_github.drop_duplicates(subset='url', inplace=True)

# Fix: Fill missing descriptions
df_github['description'].replace('', 'No description available', inplace=True)
df_github['description'].fillna('No description available', inplace=True)

# Fix: Flag incomplete rows
df_github['is_complete'] = df_github[['product_name', 'description', 'url']].apply(
    lambda row: all(row != '') and all(row.notna()), axis=1
)

print(f"\nAfter cleaning:")
print(f"  Total records        : {len(df_github)}")
print(f"  Complete records     : {df_github['is_complete'].sum()}")

# Save final cleaned version
df_github.to_csv("github_marketplace_cleaned.csv", index=False)
print("\n✅ Saved to 'github_marketplace_cleaned.csv'")
df_github.head(5)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


PART 1: Scraping GitHub Marketplace Actions...
Scraping page 1 | Products so far: 0
No cards found on page 1. Trying alternative selector...
  ✓ Found 1 products on page 1
Scraping page 2 | Products so far: 1
No cards found on page 2. Trying alternative selector...
  ✓ Found 1 products on page 2
Scraping page 3 | Products so far: 2
No cards found on page 3. Trying alternative selector...
  ✓ Found 1 products on page 3
Scraping page 4 | Products so far: 3
No cards found on page 4. Trying alternative selector...
  ✓ Found 1 products on page 4
Scraping page 5 | Products so far: 4
No cards found on page 5. Trying alternative selector...
  ✓ Found 1 products on page 5
Scraping page 6 | Products so far: 5
No cards found on page 6. Trying alternative selector...
  ✓ Found 1 products on page 6
Scraping page 7 | Products so far: 6
No cards found on page 7. Trying alternative selector...
  ✓ Found 1 products on page 7
Scraping page 8 | Products so far: 7
No cards found on page 8. Trying alternat

/tmp/ipython-input-472/1298992343.py:164: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_github['description'].replace('', 'No description available', inplace=True)
/tmp/ipython-input-472/1298992343.py:165: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df

,product_name,description,url,page_number,cleaned_name,cleaned_description,is_complete
0,Search syntax tips,No description available,https://docs.github.com/search-github/github-c...,1,search syntax tip,,True


#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [8]:
# ============================================================
# Question 5: Scrape Tweets using Tweepy API
# ============================================================

!pip install tweepy pandas nltk

import tweepy
import pandas as pd
import re
import time
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# -----------------------------------------------
# PART 1: Authenticate & Scrape Tweets
# -----------------------------------------------
# 🔑 Replace with your actual Twitter API keys from developer.twitter.com

BEARER_TOKEN      = "YOUR_BEARER_TOKEN"
API_KEY           = "YOUR_API_KEY"
API_KEY_SECRET    = "YOUR_API_KEY_SECRET"
ACCESS_TOKEN      = "YOUR_ACCESS_TOKEN"
ACCESS_TOKEN_SECRET = "YOUR_ACCESS_TOKEN_SECRET"

# Authenticate using Bearer Token (v2 API - free tier)
client = tweepy.Client(bearer_token=BEARER_TOKEN, wait_on_rate_limit=True)
print("✅ Authenticated with Twitter API v2")

# Search hashtags related to ML and AI
QUERIES = [
    "#MachineLearning -is:retweet lang:en",
    "#ArtificialIntelligence -is:retweet lang:en",
    "#DeepLearning -is:retweet lang:en",
    "#DataScience -is:retweet lang:en"
]

TWEETS_PER_QUERY = 250   # 4 queries × 250 = 1000 tweets (free tier limit aware)
MAX_RESULTS_PER_PAGE = 100

all_tweets = []

for query in QUERIES:
    print(f"\n🔍 Fetching tweets for: {query}")
    tweet_count = 0

    try:
        paginator = tweepy.Paginator(
            client.search_recent_tweets,
            query=query,
            tweet_fields=["id", "text", "author_id", "created_at",
                          "public_metrics", "lang"],
            user_fields=["username", "name", "public_metrics"],
            expansions=["author_id"],
            max_results=MAX_RESULTS_PER_PAGE,
            limit=3  # 3 pages × 100 = 300 per query
        )

        # Build user id → username map
        for response in paginator:
            users = {}
            if response.includes and "users" in response.includes:
                for user in response.includes["users"]:
                    users[user.id] = {
                        "username": user.username,
                        "name": user.name,
                        "followers": user.public_metrics.get("followers_count", 0)
                            if user.public_metrics else 0
                    }

            if response.data:
                for tweet in response.data:
                    user_info = users.get(tweet.author_id, {})
                    all_tweets.append({
                        "tweet_id": tweet.id,
                        "username": user_info.get("username", ""),
                        "name": user_info.get("name", ""),
                        "followers": user_info.get("followers", 0),
                        "text": tweet.text,
                        "created_at": tweet.created_at,
                        "like_count": tweet.public_metrics.get("like_count", 0)
                            if tweet.public_metrics else 0,
                        "retweet_count": tweet.public_metrics.get("retweet_count", 0)
                            if tweet.public_metrics else 0,
                        "reply_count": tweet.public_metrics.get("reply_count", 0)
                            if tweet.public_metrics else 0,
                        "query": query
                    })
                    tweet_count += 1

        print(f"  ✓ Collected {tweet_count} tweets")

    except tweepy.TooManyRequests:
        print("  ⚠️ Rate limit hit. Waiting 15 minutes...")
        time.sleep(900)

    except tweepy.TwitterServerError as e:
        print(f"  ✗ Twitter server error: {e}")

    time.sleep(5)  # Polite delay between queries

# Save raw tweets
df_tweets = pd.DataFrame(all_tweets)
df_tweets.drop_duplicates(subset="tweet_id", inplace=True)
df_tweets.reset_index(drop=True, inplace=True)
df_tweets.to_csv("tweets_raw.csv", index=False)
print(f"\n✅ Total unique tweets collected: {len(df_tweets)}")
print(df_tweets.head(3))

# -----------------------------------------------
# PART 2: Data Cleaning
# -----------------------------------------------

print("\n" + "="*60)
print("PART 2: DATA CLEANING")
print("="*60)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_tweet(text):
    if pd.isna(text): return ""
    text = re.sub(r'http\S+|www\S+', '', text)     # Remove URLs
    text = re.sub(r'@\w+', '', text)                # Remove @mentions
    text = re.sub(r'#', '', text)                   # Remove # symbol (keep word)
    text = re.sub(r'RT\s+', '', text)               # Remove RT prefix
    text = re.sub(r'[^a-zA-Z\s]', '', text)         # Remove special chars & numbers
    text = text.lower().strip()                     # Lowercase
    tokens = word_tokenize(text)                    # Tokenize
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]  # Remove stopwords
    tokens = [lemmatizer.lemmatize(t) for t in tokens]  # Lemmatize
    return ' '.join(tokens)

df_tweets['cleaned_text'] = df_tweets['text'].apply(clean_tweet)

# -----------------------------------------------
# Data Quality Check
# -----------------------------------------------
print("\n--- Data Quality Report ---")
print(f"Total tweets           : {len(df_tweets)}")
print(f"Duplicate tweet IDs    : {df_tweets['tweet_id'].duplicated().sum()}")
print(f"Missing text           : {df_tweets['text'].isna().sum()}")
print(f"Empty after cleaning   : {(df_tweets['cleaned_text'] == '').sum()}")
print(f"Missing usernames      : {(df_tweets['username'] == '').sum()}")

# Fix: Drop rows where cleaned text is empty
df_tweets = df_tweets[df_tweets['cleaned_text'] != '']
df_tweets.reset_index(drop=True, inplace=True)

# Verify all required columns are filled
required_cols = ['tweet_id', 'username', 'text', 'cleaned_text', 'created_at']
for col in required_cols:
    missing = df_tweets[col].isna().sum() + (df_tweets[col] == '').sum()
    print(f"  {col:<20}: {missing} missing values")

# Save cleaned tweets
df_tweets.to_csv("tweets_cleaned.csv", index=False)
print(f"\n✅ Final cleaned tweets: {len(df_tweets)}")
print("✅ Saved to 'tweets_cleaned.csv'")
df_tweets[['username', 'text', 'cleaned_text', 'created_at']].head(5)

✅ Authenticated with Twitter API v2

🔍 Fetching tweets for: #MachineLearning -is:retweet lang:en


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Unauthorized: 401 Unauthorized
Unauthorized

# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

Reflection on the Assignment

The assignment demonstrated complete NLP operations because it showed all processing steps from data collection to text preprocessing and syntactic analysis. The process showed me how organizations use different NLP methods to transform their raw text data into usable information.

The most difficult tasks required users to extract content from websites and establish connections with application programming interfaces. The team needed to solve three problems which included handling blocked requests and authentication errors and maintaining system performance limits. The process of building constituency parsing through custom grammars in NLTK required multiple development stages to achieve proper results.

I found Named Entity Recognition to be an interesting task because it showed real-world effects through different NLP methods which I studied through stemming and lemmatization. The process of building the text-cleaning pipeline from its initial stage to its final state brought me great pleasure.

The assignment maintained a strong structure which delivered important content yet required more time to finish because of its extensive requirements. The additional time will help users to study each idea in greater detail.

> Add blockquote

> Add blockquote





In [9]:
# ============================================================
# Download Cleaned CSV Files
# ============================================================
from google.colab import files

files.download("research_papers_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>